# Loading in packages

# Loading in and quality controlling the enrollment data

In [0]:
#Reading in the enrollment data to identify cut-off dates (based on who was under 18 when they signed up). 
!pip install openpyxl
import pandas as pd

enroll_info_path = "/Workspace/Users/user/user/UMD Enrollment Log_11.7.2025_for_Gabriel.xlsx"

enroll_info = pd.read_excel(enroll_info_path)
enroll_info.head(5)

In [0]:
# Rows with NaT in either date column
na_rows = enroll_info[enroll_info['Date of Enrollment'].isnull() | enroll_info['D.O.B.'].isnull()]
print(f"Rows with missing dates: {len(na_rows)}")
display(na_rows)

In [0]:
# Filter them out
enroll_info_filtered = enroll_info.dropna(subset=['Date of Enrollment', 'D.O.B.']).copy()
print(f"Remaining rows: {len(enroll_info_filtered)}")

In [0]:
# Count unique types per column
type_counts = {}
for col in enroll_info_filtered.columns:
    type_counts[col] = enroll_info_filtered[col].apply(type).nunique()

# Show columns with mixed types
mixed_cols = {k: v for k, v in type_counts.items() if v > 1}
print(f"Columns with mixed types: {len(mixed_cols)}")
print(mixed_cols)

# Display type breakdown for each mixed column
for col in mixed_cols:
    print(f"\n--- {col} ---")
    print(enroll_info_filtered[col].apply(type).value_counts())

In [0]:
#Seeing the rows where there are non-float values in the Age at Enrollment column
enroll_info_filtered_AoE_non_float_rows = enroll_info_filtered[enroll_info_filtered['Age at Enrollment'].apply(lambda x: not isinstance(x, float))].copy()
enroll_info_filtered_AoE_non_float_rows['Age at Enrollment'] = enroll_info_filtered_AoE_non_float_rows['Age at Enrollment'].astype(str)
display(enroll_info_filtered_AoE_non_float_rows)

In [0]:
#All values above seem reasonable to keep, especially since we are manually calculating age at enrollment later anyway
enroll_info_filtered_AoE_float = enroll_info_filtered.copy()
enroll_info_filtered_AoE_float['Age at Enrollment'] = pd.to_numeric(enroll_info_filtered_AoE_float['Age at Enrollment'], errors='coerce')
display(enroll_info_filtered_AoE_float)

In [0]:
# Double checking previously odd values to ensure they are now floats
enroll_info_filtered_AoE_previously_non_float_rows = enroll_info_filtered_AoE_float[enroll_info_filtered_AoE_float['Subject ID'].isin(enroll_info_filtered_AoE_non_float_rows['Subject ID'])]
display(enroll_info_filtered_AoE_previously_non_float_rows)

In [0]:
#checking 5 different values for both floats and strings for column Enrollment Status as a quality check
for dtype in enroll_info_filtered['Enrollment Status'].apply(type).unique():
    enrollment_status_vals = enroll_info_filtered[enroll_info_filtered['Enrollment Status'].apply(lambda x: isinstance(x, dtype))]['Enrollment Status'].head(5).tolist()
    print(f"\n--- {dtype} ---")
    print(enrollment_status_vals)

In [0]:
#Checking each unique value for the floats in that column to quality check that they are all na values
enrollment_status_float_vals = enroll_info_filtered[enroll_info_filtered['Enrollment Status'].apply(lambda x: isinstance(x, float))]['Enrollment Status'].unique()
print(enrollment_status_float_vals)

In [0]:
#Setting enroll_info_filtered_AoE_float to be just enroll_info_filtered2 for easier reference since there's no need to change the Enrollment Status values
enroll_info_filtered2 = enroll_info_filtered_AoE_float
display(enroll_info_filtered2)

In [0]:
# Look for duplicate MRNs 
duplicate_mrns = enroll_info_filtered2[enroll_info_filtered2['Epic MRN'].duplicated(keep=False)].sort_values('Epic MRN')
print(f"Rows with duplicate MRNs: {len(duplicate_mrns)}")
display(duplicate_mrns)

In [0]:
# For both pairs of duplicate MRNs, the birthday is the same for both individuals. This makes it likely that they are the same individual, just enrolled twice for some reason. Removing latter enrollment event for each. 
# Confirmed that these are duplicate entries. 

enroll_info_filtered3 = enroll_info_filtered2.sort_values('Date of Enrollment').drop_duplicates(subset='Epic MRN', keep='first').copy()
print(f"Before: {len(enroll_info_filtered2)} rows")
print(f"After: {len(enroll_info_filtered3)} rows")
# Rows in filtered2 but not in filtered3
dropped_rows_enroll_info_filtered23 = enroll_info_filtered2[~enroll_info_filtered2.index.isin(enroll_info_filtered3.index)]
display(dropped_rows_enroll_info_filtered23)

In [0]:
# Splitting those enrolled as adults and those enrolled as minors
import pandas as pd
from dateutil.relativedelta import relativedelta

# Parse as datetime
enroll_info_filtered3['Date of Enrollment'] = pd.to_datetime(enroll_info_filtered3['Date of Enrollment'], format='%Y-%m-%d')
enroll_info_filtered3['D.O.B.'] = pd.to_datetime(enroll_info_filtered3['D.O.B.'], format='%Y-%m-%d')

# Calculate exact age at enrollment using relativedelta
enroll_info_filtered3['age_at_enrollment'] = enroll_info_filtered3.apply(
    lambda row: relativedelta(row['Date of Enrollment'], row['D.O.B.']).years, axis=1
)

# Split into two dataframes (those joined after 18 and those joined before 18)
enrolled_as_adults = enroll_info_filtered3[enroll_info_filtered3['age_at_enrollment'] >= 18].copy()
enrolled_as_minors = enroll_info_filtered3[enroll_info_filtered3['age_at_enrollment'] < 18].copy()

# Add 18th birthday column for those who joined as minors
enrolled_as_minors['18th_birthday'] = enrolled_as_minors['D.O.B.'] + pd.DateOffset(years=18)

print(f"Enrolled at 18+: {len(enrolled_as_adults)} rows")
print(f"Enrolled under 18: {len(enrolled_as_minors)} rows")

display(enrolled_as_adults.head())
display(enrolled_as_minors.head())

In [0]:
# Combining enrolled_as_minors and enrolled_as_adults into enrolled_full
enrolled_full = pd.concat([enrolled_as_minors, enrolled_as_adults], ignore_index=True)
print(f"Minors: {len(enrolled_as_minors)} + Adults: {len(enrolled_as_adults)} = Total: {len(enrolled_full)}")

# Loading in and filtering notes

In [0]:
#Reading in the notes
import pandas as pd

notes_path = "/Workspace/Users/user/user/umdhpo_notes_202605141230.csv"

notes = pd.read_csv(notes_path)
notes.head(5)

In [0]:
# Filtering out those enrolled before 18 whose note was written after their 18th birthday and joining back together the qualifying portion of the cohort
import pandas as pd
from dateutil.relativedelta import relativedelta

# Ensure datetime
notes['NOTE_FILE_TIME_DTTM'] = pd.to_datetime(notes['NOTE_FILE_TIME_DTTM'])

# Left merge to keep all rows including adults
notes_merged = notes.merge(
    enrolled_as_minors[['Epic MRN', '18th_birthday']],
    left_on='PAT_MRN_ID',
    right_on='Epic MRN',
    how='left'
)

# Display the rows being removed (minors whose notes were filed after they turned 18)
filed_after_18 = notes_merged[
    notes_merged['18th_birthday'].notnull() & 
    (notes_merged['18th_birthday'] < notes_merged['NOTE_FILE_TIME_DTTM'])
]
display(filed_after_18)

# Keep everything else (adults + minors whose notes were filed before they turned 18)
notes_filtered1 = notes_merged[
    notes_merged['18th_birthday'].isnull() | 
    (notes_merged['NOTE_FILE_TIME_DTTM'] <= notes_merged['18th_birthday'])
].drop(columns=['Epic MRN', '18th_birthday']).copy()
display(notes_filtered1)

# Sanity check the math
print(f"Original notes: {len(notes)}")
print(f"Filtered notes: {len(notes_filtered1)}")
print(f"Notes filed after 18th birthday: {len(filed_after_18)}")
print(f"Confirm Original Notes - Filtered Notes - Notes filed after 18th birthday = 0:{len(notes) - len(notes_filtered1) - len(filed_after_18)}")

In [0]:
#Identifying unique values for author role to filter out irrelevant role notes. 
notes_filtered1['AUTHOR_ROLE'].value_counts()

In [0]:
#dropping rows where clinician type is not 'Physician', 'Resident', 'Nurse Practitioner', 'Fellow', or 'Medical Student' and then displaying the dropped rows
valid_roles = ['Physician', 'Resident', 'Nurse Practitioner', 'Fellow', 'Medical Student']
notes_filtered2 = notes_filtered1[notes_filtered1['AUTHOR_ROLE'].isin(valid_roles)].copy()

print(f"Before: {len(notes_filtered1)} rows")
print(f"After: {len(notes_filtered2)} rows")
dropped_roles_notes_filtered_12 = notes_filtered1[~notes_filtered1['AUTHOR_ROLE'].isin(valid_roles)]
display(dropped_roles_notes_filtered_12)

# Joining enrollment data, notes, and date of last review data then further filtering

In [0]:
# Dropping any rows from notes where there is not a matching MRN in the enrollment dataframe
# Combine all valid MRNs from both groups
valid_mrns = pd.concat([
    enrolled_full['Epic MRN']
]).unique()

# Rows being dropped (no matching MRN)
dropped_notes_matching = notes_filtered2[~notes_filtered2['PAT_MRN_ID'].isin(valid_mrns)].copy()
print(f"Rows dropped: {len(dropped_notes_matching)}")
dropped_notes_matching.head()

# Keep only notes with matching MRNs
notes_filtered3 = notes_filtered2[notes_filtered2['PAT_MRN_ID'].isin(valid_mrns)].copy()

print(f"notes_filtered2: {len(notes_filtered2)} rows")
print(f"notes_filtered3: {len(notes_filtered3)} rows")
display(notes_filtered3)

In [0]:
#Combining notes and enrollment dataframes 
notes_enrolled = notes_filtered3.merge(
    enrolled_full,
    left_on='PAT_MRN_ID',
    right_on='Epic MRN',
    how='inner'
)
print(f"Joined rows: {len(notes_enrolled)}")
display(notes_enrolled.head())

In [0]:
# Loading in redcap data
redcap_data = pd.read_csv('/Workspace/Users/user/user/UndiagnosedMendelian-EHRExtraction_DATA_LABELS_2026-05-26_1456.csv')
display(redcap_data.head())
print(len(redcap_data))

In [0]:
enrollment_log = pd.read_excel(
    "/Workspace/Users/user/user/UMD Enrollment Log_11.7.2025_for_Gabriel.xlsx",
    dtype={'Age at Enrollment': str}
)
display(enrollment_log)

In [0]:
# Merge with indicator to track matches
notes_enrolled2 = notes_enrolled.merge(
    redcap_data[['Record ID #:', 'Date of last chart review:']],
    left_on='Subject ID',
    right_on='Record ID #:',
    how='outer',
    indicator=True
)

# Fill in PAT_MRN_ID for rows that only have a Subject ID/Record ID
# by looking up the Epic MRN in enrollment_log
sid_to_mrn = enrollment_log[['Subject ID', 'Epic MRN']].dropna().copy()
sid_to_mrn['Subject ID'] = sid_to_mrn['Subject ID'].astype(str).str.strip()
sid_to_mrn['Epic MRN'] = sid_to_mrn['Epic MRN'].astype(str).str.strip()
sid_to_mrn = sid_to_mrn.drop_duplicates(subset='Subject ID')

# Use Subject ID or Record ID to look up MRN where PAT_MRN_ID is missing
lookup_id = notes_enrolled2['Subject ID'].fillna(notes_enrolled2['Record ID #:'])
lookup_id_clean = lookup_id.str.strip()
mrn_fill = lookup_id_clean.map(sid_to_mrn.set_index('Subject ID')['Epic MRN'])

notes_enrolled2['PAT_MRN_ID'] = notes_enrolled2['PAT_MRN_ID'].fillna(mrn_fill)

# Flag unmatched minors and adults
minor_ids = enrolled_as_minors['Subject ID'].unique()
adult_ids = enrolled_as_adults['Subject ID'].unique()

# MRNs that successfully matched both sides
both_mrns = set(notes_enrolled2[notes_enrolled2['_merge'] == 'both']['PAT_MRN_ID'].dropna())

# Unmatched minors in notes_enrolled
no_match_notes_minors = notes_enrolled2[
    (notes_enrolled2['_merge'] == 'left_only') & 
    (notes_enrolled2['Subject ID'].isin(minor_ids))
]
print(f"Minor notes with no REDCap match: {len(no_match_notes_minors)}")
display(no_match_notes_minors)

# Unmatched adults in notes_enrolled
no_match_notes_adults = notes_enrolled2[
    (notes_enrolled2['_merge'] == 'left_only') & 
    (notes_enrolled2['Subject ID'].isin(adult_ids))
]
print(f"Adult notes with no REDCap match: {len(no_match_notes_adults)}")
display(no_match_notes_adults)

# Unmatched REDCap rows
no_match_redcap = notes_enrolled2[
    (notes_enrolled2['_merge'] == 'right_only') & 
    (notes_enrolled2['Record ID #:'])
]
print(f"REDCap participants with no notes match: {len(no_match_redcap)}")
display(no_match_redcap)

# --- Reporting: truly lost MRNs (not in any 'both' matched row) ---
minor_mrns = set(no_match_notes_minors['PAT_MRN_ID'].dropna())
adult_mrns = set(no_match_notes_adults['PAT_MRN_ID'].dropna())
redcap_mrns = set(no_match_redcap['PAT_MRN_ID'].dropna())

print(f"\nTotal rows (nothing dropped): {len(notes_enrolled2)}")
print(f"Unique MRNs: {notes_enrolled2['PAT_MRN_ID'].nunique()}")
print(f"\nMinor notes with no REDCap match — unique MRNs: {len(minor_mrns)} (truly lost: {len(minor_mrns - both_mrns)})")
print(f"Adult notes with no REDCap match — unique MRNs: {len(adult_mrns)} (truly lost: {len(adult_mrns - both_mrns)})")
print(f"REDCap with no notes match — unique MRNs: {len(redcap_mrns)} (truly lost: {len(redcap_mrns - both_mrns)})")
print(f"\nTotal truly lost unique MRNs: {len((minor_mrns | adult_mrns | redcap_mrns) - both_mrns)}")

In [0]:
# Normalize helper for stripping spaces
def strip_clean(series):
    return series.astype(str).str.strip()

# Step 1: Build MRN -> Date of last chart review lookup from rows that have both
mrn_to_review = notes_enrolled2[notes_enrolled2['Date of last chart review:'].notna()].copy()
mrn_to_review['mrn_clean'] = strip_clean(mrn_to_review['PAT_MRN_ID'])
mrn_to_review['Date of last chart review:'] = pd.to_datetime(mrn_to_review['Date of last chart review:'])
mrn_to_review = (
    mrn_to_review
    .sort_values('Date of last chart review:', ascending=False)
    .drop_duplicates(subset='mrn_clean', keep='first')
    [['mrn_clean', 'Date of last chart review:']]
)

# Step 2: Identify unmatched notes (have MRN but missing Date of last chart review)
notes_enrolled2['mrn_clean'] = strip_clean(notes_enrolled2['PAT_MRN_ID'])
unmatched_mask = (
    notes_enrolled2['PAT_MRN_ID'].notna() & 
    notes_enrolled2['Date of last chart review:'].isna()
)

# Step 3: Fill in Date of last chart review for unmatched notes using MRN lookup
fill_values = notes_enrolled2.loc[unmatched_mask, 'mrn_clean'].map(
    mrn_to_review.set_index('mrn_clean')['Date of last chart review:']
)
notes_enrolled2.loc[unmatched_mask, 'Date of last chart review:'] = fill_values

# --- Call out rows that matched on both but still have NA in key columns ---
check_cols = ['PAT_MRN_ID', 'Date of last chart review:', 'NOTE_FILE_TIME_DTTM', 'NOTE_TEXT']

matched_both = notes_enrolled2[notes_enrolled2['_merge'] == 'both']

# MRNs that survive (have all four columns populated in at least one row)
clean_mrns = set(
    notes_enrolled2.dropna(subset=check_cols)['PAT_MRN_ID'].dropna().unique()
)

print("Unique MRNs truly lost due to NA despite matching on both:")
for col in check_cols:
    na_in_col = matched_both[matched_both[col].isna()]
    truly_lost = set(na_in_col['PAT_MRN_ID'].dropna().unique()) - clean_mrns
    print(f"  {col}: {len(truly_lost)} unique MRNs")

any_na_mask = matched_both[check_cols].isna().any(axis=1)
all_na_mrns = set(matched_both[any_na_mask]['PAT_MRN_ID'].dropna().unique())
truly_lost_total = all_na_mrns - clean_mrns
print(f"  Total truly lost: {len(truly_lost_total)} unique MRNs")

# Step 4: Drop rows that are still unmatched after the fill
notes_enrolled2_matched = notes_enrolled2.dropna(
    subset=check_cols
).copy()

notes_enrolled3 = notes_enrolled2_matched.drop(columns=['mrn_clean', '_merge']).reset_index(drop=True)

# --- Reporting ---
filled_count = fill_values.notna().sum()
filled_mrns = notes_enrolled2.loc[unmatched_mask & fill_values.notna(), 'PAT_MRN_ID'].nunique()
still_unmatched = unmatched_mask.sum() - filled_count
dropped = len(notes_enrolled2) - len(notes_enrolled3)

print(f"\nUnmatched notes filled via MRN lookup: {filled_count} rows ({filled_mrns} unique MRNs saved)")
print(f"Still unmatched after MRN lookup: {still_unmatched} rows")
print(f"Total rows dropped (no match possible): {dropped}")
print(f"\nnotes_enrolled2 rows: {len(notes_enrolled2)}")
print(f"notes_enrolled3 rows: {len(notes_enrolled3)}")
print(f"notes_enrolled2 unique MRNs: {notes_enrolled2['PAT_MRN_ID'].nunique()}")
print(f"notes_enrolled3 unique MRNs: {notes_enrolled3['PAT_MRN_ID'].nunique()}")
print(f"Unique MRNs lost: {notes_enrolled2['PAT_MRN_ID'].nunique() - notes_enrolled3['PAT_MRN_ID'].nunique()}")

In [0]:
import pandas as pd
both_missing_review = matched_both[matched_both['Date of last chart review:'].isna()]
print(f"\nMatched on both but missing Date of last chart review: {len(both_missing_review)} rows ({both_missing_review['PAT_MRN_ID'].nunique()} unique MRNs)")
pd.set_option('display.max_rows', None)
display(both_missing_review)

In [0]:
unique_ids = sorted(both_missing_review['Record ID #:'].unique())
print(f"Unique Record IDs: {len(unique_ids)}")
display(unique_ids)

## Troubleshooting and analyzing rows without matches

In [0]:
#Troubleshooting rows without matches
minor_ids = enrolled_as_minors['Subject ID'].unique()

# notes_enrolled Subject IDs not in REDCap — minors only
notes_no_match_minors = notes_enrolled[
    (~notes_enrolled['Subject ID'].isin(redcap_data['Record ID #:'])) & 
    (notes_enrolled['Subject ID'].isin(minor_ids))
]['Subject ID']
print(f"\nUnique minor Subject IDs in notes_enrolled with no match: {notes_no_match_minors.nunique()}")
print(sorted(notes_no_match_minors.unique()))

# REDCap Record IDs not in notes_enrolled — minors only
redcap_no_match_minors = redcap_data[
    (~redcap_data['Record ID #:'].isin(notes_enrolled['Subject ID'])) & 
    (redcap_data['Record ID #:'].isin(minor_ids))
]['Record ID #:']
print(f"Unique minor Record IDs in redcap_data with no match: {redcap_no_match_minors.nunique()}")
print(sorted(redcap_no_match_minors.unique()))

In [0]:
redcap_raw = pd.read_csv('/Workspace/Users/user/user/UndiagnosedMendelian-EHRExtraction_DATA_2026-05-05_1558.csv')

missing_ids = notes_no_match_minors.unique()
id_col = redcap_raw.columns[0]
found_ids = redcap_raw[redcap_raw[id_col].isin(missing_ids)][id_col].unique()

if len(found_ids) > 0:
    print(f"Found: {len(found_ids)}")
    print(sorted(found_ids))
else:
    print("None of the missing IDs were found in the raw file.")

In [0]:
#Checking author types for enrolled minors with no matching REDCap data
missing_minor_notes = notes_enrolled[notes_enrolled['Subject ID'].isin(notes_no_match_minors.unique())]
print(missing_minor_notes['AUTHOR_ROLE'].value_counts())

In [0]:
# Identifying duplicates for Subject ID in notes_enrolled2
id_counts = notes_enrolled2['Subject ID'].value_counts()
duplicates = id_counts[id_counts > 1]
print(f"Subject IDs with multiple notes: {len(duplicates)}")
print(duplicates)

Need to decide what to do with duplicates. This file should be as comparable as possible to what Rebecca loaded in so using that to determine next steps for this. 

Was using REDCap_mapping_full_path = "/Workspace/Users/user/user/UMD Data for HPO Phecode mapping.csv" then replaced with current data (previous version was outdated) - see more in notebook UMD Data Labels Harmonization

In [0]:
# Read in scrambled patient data
REDCap_mapping_full_path = "/Workspace/Users/user/user/data_labels_filtered4.csv"

REDCap_mapping = pd.read_csv(REDCap_mapping_full_path)
print('Number of rows:', len(REDCap_mapping))
print('Number of columns:', len(REDCap_mapping.columns))
display(REDCap_mapping)

In [0]:
# Identifying duplicates for Record ID # in REDCap_mapping
REDCap_id_counts = REDCap_mapping['Record ID #:'].value_counts()
REDCap_duplicates = REDCap_id_counts[REDCap_id_counts > 1]
print(f"Record ID #s with multiple notes: {len(REDCap_duplicates)}")
REDCap_duplicates

In [0]:
# Identifying duplicates for Record ID # in redcap_data
redcap_data_id_counts = redcap_data['Record ID #:'].value_counts()
redcap_data_duplicates = redcap_data_id_counts[redcap_data_id_counts > 1]
print(f"Record ID #s with multiple notes: {len(redcap_data_duplicates)}")
redcap_data_duplicates

In [0]:
# For all duplicated Subject ID values in notes_enrolled2, how many unique values for column NOTE_FILE_TIME_DTTM does each unique duplicated Subject ID have?
dup_ids = notes_enrolled2['Subject ID'].value_counts()
dup_ids = dup_ids[dup_ids > 1].index

unique_dates = (
    notes_enrolled2[notes_enrolled2['Subject ID'].isin(dup_ids)]
    .groupby('Subject ID')['NOTE_FILE_TIME_DTTM']
    .nunique()
    .sort_values(ascending=False)
)
print(unique_dates)

## Dropping all notes except note right before most recent review from notes_enrolled2

In [0]:
# Ensure both columns are datetime for proper comparison
notes_enrolled3['NOTE_FILE_TIME_DTTM'] = pd.to_datetime(notes_enrolled3['NOTE_FILE_TIME_DTTM'])
notes_enrolled3['Date of last chart review:'] = pd.to_datetime(notes_enrolled3['Date of last chart review:'])

# Only keep rows where the note was filed on or before the chart review date
before_review = notes_enrolled3[
    notes_enrolled3['NOTE_FILE_TIME_DTTM'] <= notes_enrolled3['Date of last chart review:']
].copy()

# For each PAT_MRN_ID, keep the row with the most recent NOTE_FILE_TIME_DTTM
# (i.e., the closest date right before the chart review)
notes_enrolled4 = (
    before_review
    .sort_values('NOTE_FILE_TIME_DTTM', ascending=False)
    .drop_duplicates(subset='PAT_MRN_ID', keep='first')
    .reset_index(drop=True)
)

# --- Reporting ---
notes_enrolled3_mrns = set(notes_enrolled3['PAT_MRN_ID'].dropna().unique())
notes_enrolled4_mrns = set(notes_enrolled4['PAT_MRN_ID'].dropna().unique())
lost_mrns = notes_enrolled3_mrns - notes_enrolled4_mrns

print(f"notes_enrolled3 rows: {len(notes_enrolled3)}")
print(f"notes_enrolled3 unique MRNs: {len(notes_enrolled3_mrns)}")
print(f"Rows before review date: {len(before_review)}")
print(f"notes_enrolled4 rows: {len(notes_enrolled4)}")
print(f"notes_enrolled4 unique MRNs: {len(notes_enrolled4_mrns)}")
print(f"Unique MRNs lost: {len(lost_mrns)}")

In [0]:
# Checking to see if any MRNs lost to having their notes written after the date of last review
# have another entry with a later date of last review they could be matched on

# Get the notes for lost MRNs (all their notes were after their assigned review date)
notes_enrolled3_mrns = set(notes_enrolled3['PAT_MRN_ID'].dropna().unique())
notes_enrolled4_mrns = set(notes_enrolled4['PAT_MRN_ID'].dropna().unique())
lost_mrns = notes_enrolled3_mrns - notes_enrolled4_mrns

lost_notes = notes_enrolled3[
    notes_enrolled3['PAT_MRN_ID'].isin(lost_mrns)
]

# Get ALL review dates per MRN from notes_enrolled2 (before any dropping)
all_review_dates = notes_enrolled2[notes_enrolled2['Date of last chart review:'].notna()].copy()
all_review_dates['mrn_clean'] = strip_clean(all_review_dates['PAT_MRN_ID'])
all_review_dates['Date of last chart review:'] = pd.to_datetime(all_review_dates['Date of last chart review:'])
all_review_dates = all_review_dates[['mrn_clean', 'Date of last chart review:']].drop_duplicates()

# For each lost MRN, check if any later review date would allow a note to qualify
saveable = []
for mrn in lost_mrns:
    mrn_clean = strip_clean(pd.Series([mrn])).iloc[0]
    patient_notes = lost_notes[lost_notes['PAT_MRN_ID'] == mrn]['NOTE_FILE_TIME_DTTM']
    review_dates = all_review_dates[all_review_dates['mrn_clean'] == mrn_clean]['Date of last chart review:']
    
    for review_date in review_dates:
        qualifying_notes = patient_notes[patient_notes <= review_date]
        if len(qualifying_notes) > 0:
            saveable.append({
                'PAT_MRN_ID': mrn,
                'current_review_date': lost_notes[lost_notes['PAT_MRN_ID'] == mrn]['Date of last chart review:'].iloc[0],
                'alternative_review_date': review_date,
                'qualifying_notes_count': len(qualifying_notes),
                'closest_note': qualifying_notes.max()
            })

saveable_df = pd.DataFrame(saveable)
saveable_mrns = saveable_df['PAT_MRN_ID'].nunique() if len(saveable_df) > 0 else 0

print(f"Lost MRNs: {len(lost_mrns)}")
print(f"MRNs saveable with a later review date: {saveable_mrns}")
print(f"Still lost (no option): {len(lost_mrns) - saveable_mrns}")

if len(saveable_df) > 0:
    display(saveable_df)

In [0]:
print(f'Total unique MRNs started with:', notes_enrolled2['PAT_MRN_ID'].nunique())
print(f'Total unique MRNs after all filtering:', notes_enrolled4['PAT_MRN_ID'].nunique())
print(f'Total unique MRNs lost: {notes_enrolled2["PAT_MRN_ID"].nunique() - notes_enrolled4["PAT_MRN_ID"].nunique()}')

# Loading in and filtering checklist items

Previously used checklist_full_path = "/Workspace/Users/user/user/UMD Phenotype Terms v4_final.csv" but this was missing some new conditions we wanted to include - see more in notebook UMD Data Labels Harmonization

In [0]:
#Reading in the checklist of what features to search for
import pandas as pd

checklist_full_path = "/Workspace/Users/user/user/hposetta_stone2.csv"

checklist_full = pd.read_csv(checklist_full_path)
checklist_full.head(5)

### Rebecca's code applied (with minor changes)

In [0]:
import numpy as np
import pandas as pd

# clean up csv file by dropping all of the rows that we have decided to ignore
checklist_filtered = checklist_full[checklist_full['Ignore or not'] != 'ignore'].copy()
print(f"Before: {len(checklist_full)} rows")
print(f"After: {len(checklist_filtered)} rows")

# changing first column name to "Text from Checklist"
checklist_filtered.rename(columns={'HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11':'Text from Checklist'}, inplace=True)

# Get rid of the "Ignore or not" column
checklist_filtered.drop(columns=['Ignore or not'], inplace=True)
display(checklist_filtered)

In [0]:
i = 0 #row
j = 0 #column

# reformatting redcap file so that each row is a patient and successive columns contain phenotype
records = []
for i in range(0,len(REDCap_mapping)):
    phen = []
    # Rebecca had: for j in range(0,130): but this would exclude the last column?
    for j in range(0,len(REDCap_mapping.columns)):
        if pd.isnull(REDCap_mapping.iloc[i][j]) == False: 
            phen.append(REDCap_mapping.iloc[i][j])
    records.append(phen)

records_formatted = pd.DataFrame(records)
display(records_formatted)

In [0]:
# from the dataframe above, creates a list 
    #Rebecca's code:
    # list = []
    # for j in range(0,1062): # rows - drops a row
    #     for i in range(0,23): # columns - drops two columns. Probably because last two columns are empty. 
    #         list.append(records1[i][j])       
manual_list = []
for j in range(0, len(records_formatted)): # rows
     for i in range(0, len(records_formatted.columns)): # columns
         manual_list.append(records_formatted[i][j])       

manual_list = [x for x in manual_list if x is not None]

display(manual_list)

In [0]:
print(checklist_filtered.columns)

In [0]:
print(checklist_filtered.columns)

In [0]:
hpocodes = []
for i in range(0, len(manual_list)):
    if manual_list[i][:3] == 'WES':
        id = manual_list[i]
    for j in range(0,len(checklist_filtered)): 
        # check if value is equal to "Term from checklist"
        if checklist_filtered.iloc[j][1] == (manual_list[i]): 
            #print(id) 
            #print(checklist_filtered.iloc[j][3])
            #print(id,checklist_filtered.iloc[j][3])
            manual_score = print(id, checklist_filtered.iloc[j][3])     

display(manual_score)

In [0]:
# Rebecca's Code:
#hpocodes_manual = []
#for i in range(0, len(manual_list)):
#    if manual_list[i][:3] == 'WES':
#        id = manual_list[i]
#    for j in range(0,len(checklist_filtered)): 
#        # check if value is equal to "Term from checklist"
#        if checklist_filtered.iloc[j][1] == (manual_list[i]): 
#            #print(id) 
#            #print(checklist_filtered.iloc[j][3])
#            print(id,checklist_filtered.iloc[j][3])
# Note: to create output file, uncomment "print(id)" and "print(hpo.iloc[j][3])" one at a time
# while the rest are commented out (for example, uncomment "print(id)" and comment out print(hpo.iloc[j][3])
# and print(id,hpo.iloc[j][3])) to print just the column of patient IDs, then do the same for HPO terms)



# Creates list of (Subject ID, HPO code) tuples
hpocodes_manual = []
# Loops through each item in manual_list; updates the current patient ID whenever a "WES" value appears
for i in range(0, len(manual_list)):
    if manual_list[i].startswith('WES'):
        id = manual_list[i]
    # For each item in manual_list, searches checklist_filtered for a matching term and appends the corresponding HPO code
    for j in range(0, len(checklist_filtered)):
        if checklist_filtered.iloc[j]['Term from checklist'] == manual_list[i]:
            hpocodes_manual.append((id, checklist_filtered.iloc[j]['HPO term (ID)']))

# Converts this list to a dataframe then displays it
manual_score = pd.DataFrame(hpocodes_manual, columns=['Subject ID', 'HPO term (ID)'])
display(manual_score)

In [0]:
rows = []
current_id = None
for item in manual_list:
    if item.startswith('WES'):
        current_id = item
    else:
        rows.append((current_id, item))

manual_df = pd.DataFrame(rows, columns=['Subject ID', 'Phenotype'])

manual_score = manual_df.merge(
    checklist_filtered[['Term from checklist', 'HPO term (ID)']],
    left_on='Phenotype',
    right_on='Term from checklist',
    how='inner'
).drop(columns=['Term from checklist'])
display(manual_score)

In [0]:
# Normalize phenotype names — strip whitespace once so downstream joins are clean
manual_score['Phenotype'] = manual_score['Phenotype'].str.strip()

In [0]:
manual_score.to_csv("/Workspace/Users/user/user/manual_score.csv", index=False)
print("Saved manual_score.csv")

# Preparing AI Extraction 

In [0]:
#Extracting unique phenotypes
print('Number of unique phenotypes:', checklist_filtered['Term from checklist'].nunique())
for term in checklist_filtered['Term from checklist'].unique():
    print(term)

^ column "nan" is dropped from here in prompt cell

In [0]:
display(checklist_filtered)

In [0]:
notes_enrolled4.to_csv("/Workspace/Users/user/user/notes_enrolled4.csv", index=False)
print("Saved notes_enrolled4.csv")

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list("my-agent-secrets")

# GPT Mini Full Run

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list("my-agent-secrets")

In [0]:
import openai
import pandas as pd
from openai import AzureOpenAI

import os
import pandas as pd
import numpy as np
import time
import json
import ast
import copy
from IPython.display import IFrame
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import re

In [0]:
api_key = dbutils.secrets.get(scope="my-agent-secrets", key='azure-openai-mini-key')
api_version = "2025-01-01-preview"
#api_version = "2026-03-17"
azure_endpoint = dbutils.secrets.get(scope="my-agent-secrets", key='azure-openai-mini-endpoint')  
model = "gpt-5.4-mini-pact"

In [0]:
from openai import OpenAI

client = AzureOpenAI(
    api_key = api_key,
    api_version = api_version,
    azure_endpoint = azure_endpoint
)

In [0]:
# Test 1: Check your client config
print(f"API Key set: {bool(client.api_key)}")
print(f"API Key (last 4): ...{client.api_key[-4:]}")
print(f"Base URL: {client.base_url}")
print(f"API Version: {client._api_version}")


In [0]:
# Test 2: Send a simple test message
try:
    response = client.chat.completions.create(
        model = model,
        messages=[{"role": "user", "content": "Say hello"}]
    )
    print(f"\nConnection successful!")
    print(f"Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"\nConnection failed:")
    print(f"Error type: {type(e).__name__}")
    print(f"Error: {e}")

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""



#results_mini_full = []

total_prompt_tokens = 0
total_completion_tokens = 0

#for idx, row in notes_enrolled4.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT']
    
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    
    total_prompt_tokens += response.usage.prompt_tokens
    total_completion_tokens += response.usage.completion_tokens
    
    try:
        output = json.loads(response.choices[0].message.content)
        phenos = output.get("Phenotypes", {})
        for pheno_name, details in phenos.items():
            results_mini_full.append({
                "Subject ID": subject_id,
                "Phenotype": pheno_name,
                "Score": details.get("score", 0),
                "Justification": details.get("justification", ""),
                "Quote": details.get("quote", "")
            })
    except json.JSONDecodeError:
        print(f"Failed to parse response for {subject_id} (row {idx})")
        results_mini_full.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

print(f"Prompt tokens: {total_prompt_tokens:,}")
print(f"Completion tokens: {total_completion_tokens:,}")
print(f"Total tokens: {total_prompt_tokens + total_completion_tokens:,}")

# Save results_mini_full — now in long format (one row per subject × phenotype)
results_mini_full_df = pd.DataFrame(results_mini_full)
#results_mini_full_df.to_json(
    #"/Workspace/Users/user/user/UMD_HPO_GPT_Mini_Output_Results_full.json",
    orient="records", indent=2
)
print(f"Saved {len(results_mini_full_df)} rows")
display(results_mini_full_df.head(10))

Prompt tokens: 3,952,414
Completion tokens: 2,730,073
Total tokens: 6,682,487
Saved 83501 rows
2 hours, 52 minutes

In [0]:
import pandas as pd

results_mini_full_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_GPT_Mini_Output_Results_full.json", orient="records")
results_mini_full_df['Phenotype'] = results_mini_full_df['Phenotype'].str.strip()
display(results_mini_full_df)
print(len(results_mini_full_df))

manual_score = pd.read_csv("/Workspace/Users/user/user/manual_score.csv")
display(manual_score)
print(len(manual_score))

notes_enrolled4 = pd.read_csv("/Workspace/Users/user/user/notes_enrolled4.csv")
display(notes_enrolled4)
print(len(notes_enrolled4))


In [0]:
subjects_incomplete = (
    results_mini_full_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
#display(subjects_incomplete)

for subject_id in subjects_incomplete['Subject ID']:
    print(f"\nSubject ID: {subject_id}")
    display(results_mini_full_df[results_mini_full_df['Subject ID'] == subject_id])

In [0]:
subjects_excess = (
    results_mini_full_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x > 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with more than 125 phenotypes scored: {len(subjects_excess)}")
#display(subjects_excess)

for subject_id in subjects_excess['Subject ID']:
    print(f"\nSubject ID: {subject_id}")
    display(results_mini_full_df[results_mini_full_df['Subject ID'] == subject_id])

In [0]:
print(f"notes_enrolled4 subjects: {notes_enrolled4['Subject ID'].nunique()}")
print(f"results_mini_full_df subjects: {results_mini_full_df['Subject ID'].nunique()}")

In [0]:
phenotypes_results = set(results_mini_full_df['Phenotype'].dropna().unique())
phenotypes_manual = set(manual_score['Phenotype'].dropna().unique())

missing_from_manual = sorted(phenotypes_results - phenotypes_manual)
missing_from_results = sorted(phenotypes_manual - phenotypes_results)

print(len(missing_from_manual), "values in results_mini_full_df but not in manual_score")
print(missing_from_manual)

print()

print(len(missing_from_results), "values in manual_score but not in results_mini_full_df")
print(missing_from_results)

In [0]:
phenotypes = set(results_mini_full_df['Phenotype'].dropna().unique())
checklist_terms = set(checklist_filtered['Term from checklist'].dropna().unique())

missing = sorted(phenotypes - checklist_terms)
print(len(missing), "values in Phenotype but not in the checklist")
missing

In [0]:
remap = {
    # Parenthesis fixes — all should have closing )
    'Double Outlet Right Ventricle (DORV': 'Double Outlet Right Ventricle (DORV)',
    'Hypertrophic cardiomyopathy (HCM': 'Hypertrophic cardiomyopathy (HCM)',
    'Tetralogy of Fallot (TOF': 'Tetralogy of Fallot (TOF)',
    'CPEO (opthalmoplegia': 'CPEO (opthalmoplegia)',
    'Autism Spectrum Disorder (ASD': 'Autism Spectrum Disorder (ASD)',
    'Diabetes Mellitus Type I (insulin dependent': 'Diabetes Mellitus Type I (insulin dependent)',
    'Diabetes Mellitus Type II (non-insulin dependent': 'Diabetes Mellitus Type II (non-insulin dependent)',
    # Hallucination fix
    'Chiari I malformation': 'Chiari malformation',
}

manual_score['Phenotype'] = manual_score['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
results_mini_full_df['Phenotype'] = results_mini_full_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
checklist_filtered['Term from checklist'] = checklist_filtered['Term from checklist'].str.strip().map(lambda x: remap.get(x, x))

In [0]:
phenotypes = set(results_mini_full_df['Phenotype'].dropna().unique())
checklist_terms = set(checklist_filtered['Term from checklist'].dropna().unique())

missing = sorted(phenotypes - checklist_terms)
print(len(missing), "values in AI Phenotype but not in the checklist")
missing

In [0]:
phenotypes = set(manual_score['Phenotype'].dropna().unique())
checklist_terms = set(checklist_filtered['Term from checklist'].dropna().unique())

missing = sorted(phenotypes - checklist_terms)
print(len(missing), "values in manual Phenotype but not in the checklist")
missing

In [0]:
phenotypes = set(results_mini_full_df['Phenotype'].dropna().unique())
checklist_terms = set(checklist_filtered['Term from checklist'].dropna().unique())

missing = sorted(checklist_terms - phenotypes)
print(len(missing), "values in checklist but not in the AI Phenotype")
missing

In [0]:
phenotypes = set(manual_score['Phenotype'].dropna().unique())
checklist_terms = set(checklist_filtered['Term from checklist'].dropna().unique())

missing = sorted(checklist_terms - phenotypes)
print(len(missing), "values in checklist but not in the AI Phenotype")
missing

In [0]:
phenotypes_results = set(results_mini_full_df['Phenotype'].dropna().unique())
phenotypes_manual = set(manual_score['Phenotype'].dropna().unique())

missing_from_manual = sorted(phenotypes_results - phenotypes_manual)
missing_from_results = sorted(phenotypes_manual - phenotypes_results)

print(len(missing_from_manual), "values in results_mini_full_df but not in manual_score")
print(missing_from_manual)

print()

print(len(missing_from_results), "values in manual_score but not in results_mini_full_df")
print(missing_from_results)

In [0]:
all_terms = sorted(set(results_mini_full_df['Phenotype'].dropna().unique()) |
                   set(manual_score['Phenotype'].dropna().unique()) |
                   set(checklist_filtered['Term from checklist'].dropna().unique()))

comparison = []
for term in all_terms:
    comparison.append({
        'Term': term,
        'In results_mini_full_df': term in set(results_mini_full_df['Phenotype'].dropna().unique()),
        'In manual_score': term in set(manual_score['Phenotype'].dropna().unique()),
        'In checklist_filtered': term in set(checklist_filtered['Term from checklist'].dropna().unique())
    })

comparison_df = pd.DataFrame(comparison)
display(comparison_df)

In [0]:
comparison_df[comparison_df[['In results_mini_full_df', 'In manual_score', 'In checklist_filtered']].eq(False).any(axis=1)]

In [0]:
phenotypes_results = set(results_mini_full_df['Phenotype'].dropna().unique())
phenotypes_manual = set(manual_score['Phenotype'].dropna().unique())

missing_from_manual = sorted(phenotypes_results - phenotypes_manual)
missing_from_results = sorted(phenotypes_manual - phenotypes_results)

print(len(missing_from_manual), "values in results_mini_full_df but not in manual_score")
print(missing_from_manual)

print()

print(len(missing_from_results), "values in manual_score but not in results_mini_full_df")
print(missing_from_results)

In [0]:
phenotypes_results = set(results_mini_full_df['Phenotype'].dropna().unique())
phenotypes_checklist = set(checklist_filtered['Term from checklist'].dropna().unique())

only_in_results = sorted(phenotypes_results - phenotypes_checklist)
only_in_checklist = sorted(phenotypes_checklist - phenotypes_results)

print("Only in results_mini_full_df:", only_in_results)
print("Only in checklist_filtered:", only_in_checklist)

In [0]:
sorted(checklist_filtered['Term from checklist'].dropna().unique().tolist())

In [0]:
sorted(results_mini_full_df['Phenotype'].dropna().unique().tolist())

In [0]:
subjects_incomplete = (
    results_mini_full_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
#display(subjects_incomplete)

for subject_id in subjects_incomplete['Subject ID']:
    print(f"\nSubject ID: {subject_id}")
    display(results_mini_full_df[results_mini_full_df['Subject ID'] == subject_id])

In [0]:
subjects_excess = (
    results_mini_full_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x > 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_excess)}")
#display(subjects_excess)

for subject_id in subjects_excess['Subject ID']:
    print(f"\nSubject ID: {subject_id}")
    display(results_mini_full_df[results_mini_full_df['Subject ID'] == subject_id])

In [0]:
# Filter manual_score to only Subject IDs in notes_enrolled4
manual_in_notes = manual_score[manual_score['Subject ID'].isin(notes_enrolled4['Subject ID'])]

# Check for duplicate Subject ID + phenotype pairs
dupes = manual_in_notes[manual_in_notes.duplicated(subset=['Subject ID', 'Phenotype'], keep=False)]

print(f"Duplicate rows: {len(dupes)}")
print(f"Across {dupes['Subject ID'].nunique()} unique Subject IDs")
if not dupes.empty:
    display(dupes.sort_values(['Subject ID', 'Phenotype']))

In [0]:
# Flag present phenotypes for each source
ai_flagged = (
    results_mini_full_df
    .query('Score == 1')[['Subject ID', 'Phenotype']]
    .drop_duplicates()
    .assign(AI=1)
)

manual_flagged = (
    manual_score[manual_score['Subject ID'].isin(notes_enrolled4['Subject ID'])]
    [['Subject ID', 'Phenotype']]
    .assign(Phenotype=lambda df: df['Phenotype'].str.strip())
    .drop_duplicates()
    .assign(Manual=1)
)

manual_raw = (
    manual_score[manual_score['Subject ID'].isin(notes_enrolled4['Subject ID'])]
    [['Subject ID', 'Phenotype']]
    .assign(Phenotype=lambda df: df['Phenotype'].str.strip())
)
manual_flagged = manual_raw.drop_duplicates().assign(Manual=1)

print(f"Manual duplicates dropped: {len(manual_raw) - len(manual_flagged)}")

ai_raw = (
    results_mini_full_df
    .query('Score == 1')[['Subject ID', 'Phenotype']]
)
ai_flagged = ai_raw.drop_duplicates().assign(AI=1)

print(f"AI duplicates dropped: {len(ai_raw) - len(ai_flagged)}")

print(f"Total AI phenotypes (should be 125 or less): {ai_flagged['Phenotype'].nunique()}")
print(f"Total manual phenotypes (should be 125 or less): {manual_flagged['Phenotype'].nunique()}")
print(f"AI flagged: {len(ai_flagged)} | Manual flagged: {len(manual_flagged)}")
print(f"Manual Shape: {manual_flagged.shape}")
print(f"AI Shape: {ai_flagged.shape}")

display(ai_flagged)
display(manual_flagged)

In [0]:
ai_raw[ai_raw.duplicated(keep=False)]

In [0]:
display(results_mini_full_df[results_mini_full_df.duplicated(subset=['Subject ID', 'Phenotype'], keep=False)])

In [0]:
results_mini_full_df = results_mini_full_df.drop_duplicates()
print(len(results_mini_full_df))

In [0]:
results_mini_full_df.to_json("/Workspace/Users/user/user/UMD_HPO_GPT_Mini_Output_Results_full_final.json", orient="records", indent=2)

In [0]:
# Only include subjects that appear in BOTH datasets
shared_subjects = sorted(set(ai_flagged['Subject ID']) & set(manual_flagged['Subject ID']))
phenotypes = sorted(set(
    results_mini_full_df['Phenotype'].dropna().tolist() +
    manual_flagged['Phenotype'].dropna().tolist()
))

print(f"Shared subjects: {len(shared_subjects)}")
print(f"Phenotypes: {len(phenotypes)}")
print(f"Expected comparisons: {len(shared_subjects)} x {len(phenotypes)} = {len(shared_subjects) * len(phenotypes)}")

# Full grid via cross join, then merge in flags (missing = 0)
comparison = (
    pd.DataFrame({'Subject ID': shared_subjects})
    .merge(pd.DataFrame({'Phenotype': phenotypes}), how='cross')
    .merge(manual_flagged, on=['Subject ID', 'Phenotype'], how='left')
    .merge(ai_flagged, on=['Subject ID', 'Phenotype'], how='left')
    .fillna({'Manual': 0, 'AI': 0})
    .astype({'Manual': int, 'AI': int})
)
comparison['match'] = comparison['Manual'] == comparison['AI']

print(f"\nTotal comparisons: {len(comparison)}")
print(f"Matches: {comparison['match'].sum()}")
print(f"Mismatches: {(~comparison['match']).sum()}")
print(f"Agreement rate: {comparison['match'].mean():.1%}")

In [0]:
# Extract only the rows where manual and AI scores disagree
# Manual=1, AI=0 means the human flagged it but the AI missed it
# Manual=0, AI=1 means the AI flagged it but the human did not
diffs = comparison[~comparison['match']][['Subject ID', 'Phenotype', 'Manual', 'AI']].copy()
diffs = diffs.sort_values(['Subject ID', 'Phenotype']).reset_index(drop=True)

print(f"Total mismatches: {len(diffs)}")
print(f"Across {diffs['Subject ID'].nunique()} unique Subject IDs")
display(diffs)

In [0]:
# For each mismatch, pull in the AI's justification and the original clinical note
mismatch_ids = diffs['Subject ID'].unique()

# AI justification — merge on BOTH Subject ID and Phenotype for phenotype-specific match
justifications = results_mini_full_df[
    results_mini_full_df['Subject ID'].isin(mismatch_ids)
][['Subject ID', 'Phenotype', 'Justification', 'Quote']]

# Original clinical note
notes = notes_enrolled4[notes_enrolled4['Subject ID'].isin(mismatch_ids)][['Subject ID', 'NOTE_TEXT']]

# Merge on both keys so each row gets only its own justification
diffs_detail = diffs.merge(justifications, on=['Subject ID', 'Phenotype'], how='left')
diffs_detail = diffs_detail.merge(notes, on='Subject ID', how='left')

diffs_detail = diffs_detail[['Subject ID', 'Phenotype', 'Manual', 'AI', 'Justification', 'Quote', 'NOTE_TEXT']]
diffs_detail = diffs_detail.sort_values(['Subject ID', 'Phenotype']).reset_index(drop=True)

print(f"Mismatch details: {len(diffs_detail)} rows across {len(mismatch_ids)} subjects")
print(len(set(results_mini_full_df['Subject ID']) & set(manual_score['Subject ID'])))   # all processed subjects in both
print(len(shared_subjects))                                                     # what you actually compared
display(diffs_detail)

In [0]:
subject_stats_df = comparison.groupby('Subject ID').agg(
    Manual_Total=('Manual', 'sum'),
    AI_Total=('AI', 'sum'),
    Shared=('match', lambda x: ((comparison.loc[x.index, 'Manual'] == 1) & (comparison.loc[x.index, 'AI'] == 1)).sum()),
).reset_index()
subject_stats_df['Manual_Only'] = subject_stats_df['Manual_Total'] - subject_stats_df['Shared']
subject_stats_df['AI_Only'] = subject_stats_df['AI_Total'] - subject_stats_df['Shared']
subject_stats_df = subject_stats_df[['Subject ID', 'Manual_Total', 'Manual_Only', 'Shared', 'AI_Only', 'AI_Total']]
display(subject_stats_df)

In [0]:
merged = manual_flagged[['Subject ID', 'Phenotype']].merge(
    ai_flagged[['Subject ID', 'Phenotype']], 
    on=['Subject ID', 'Phenotype'], 
    how='outer', 
    indicator=True
)

subject_stats_df = merged.groupby('Subject ID')['_merge'].value_counts().unstack(fill_value=0)
print(subject_stats_df)
subject_stats_df = subject_stats_df.rename(columns={
    'left_only': 'Manual Only',
    'both': 'Shared',
    'right_only': 'AI Only'
})
subject_stats_df['Manual Total'] = subject_stats_df['Manual Only'] + subject_stats_df['Shared']
subject_stats_df['AI Total'] = subject_stats_df['AI Only'] + subject_stats_df['Shared']
subject_stats_df = subject_stats_df[['Manual Total', 'Manual Only', 'Shared', 'AI Only', 'AI Total']].reset_index()
display(subject_stats_df)

In [0]:
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(comparison['Manual'], comparison['AI'])
print(f"Cohen's Kappa: {kappa:.4f}")

In [0]:
print("Manual '1' counts:", (comparison['Manual'] == 1).sum())
print("Manual '0' counts:", (comparison['Manual'] == 0).sum())
print("GPT mini '1' counts:", (comparison['AI'] == 1).sum())
print("GPT mini '0' counts:", (comparison['AI'] == 0).sum())
print("Total comparisons:", len(comparison))

In [0]:
print("Model: GPT 5.4 - mini")
total_notes = comparison['Subject ID'].nunique()
total_phenotypes = comparison['Phenotype'].nunique()
total_scores = total_notes * total_phenotypes

both_1 = ((comparison['Manual'] == 1) & (comparison['AI'] == 1)).sum()
both_0 = ((comparison['Manual'] == 0) & (comparison['AI'] == 0)).sum()
ai_1_manual_0 = ((comparison['AI'] == 1) & (comparison['Manual'] == 0)).sum()
manual_1_ai_0 = ((comparison['Manual'] == 1) & (comparison['AI'] == 0)).sum()

print(f"Total notes scored: {total_notes}")
print(f"Total phenotypes: {total_phenotypes}")
print(f"Total scores: {total_notes} x {total_phenotypes} = {total_scores}")
print(f"Number of rows (should be equal to total scores): {len(comparison)}")
print(f"Both scored 1: {both_1} ({both_1/total_scores*100:.1f}%)")
print(f"Both scored 0: {both_0} ({both_0/total_scores*100:.1f}%)")
print(f"AI=1, Manual=0: {ai_1_manual_0} ({ai_1_manual_0/total_scores*100:.1f}%)")
print(f"Manual=1, AI=0: {manual_1_ai_0} ({manual_1_ai_0/total_scores*100:.1f}%)")